# Results Visualization and Reporting

This notebook creates comprehensive visualizations and reports for the churn prediction model results.

In [ ]:
# Install required packages
%pip install pandas matplotlib seaborn plotly

# Imports
import pandas as pd
import numpy as np
from pathlib import Path
import pickle

import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

In [ ]:
# Configuration
PROCESSED_DIR = Path("../data/processed")
MODELS_DIR = Path("../models")
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Load model comparison results
print("=== Loading Model Results ===")

results_df = pd.read_csv(MODELS_DIR / "model_comparison_results.csv", index_col=0)
print("Model comparison results loaded:")
print(results_df.round(4))

In [ ]:
# Comprehensive model performance dashboard
print("=== Model Performance Dashboard ===")

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Test Accuracy
results_df['test_accuracy'].plot(kind='bar', ax=axes[0, 0], color='steelblue')
axes[0, 0].set_title('Test Accuracy', fontsize=14, fontweight='bold')
axes[0, 0].set_ylabel('Accuracy', fontsize=12)
axes[0, 0].tick_params(axis='x', rotation=45)
axes[0, 0].axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Baseline')
axes[0, 0].legend()

# Test ROC-AUC
results_df['test_roc_auc'].plot(kind='bar', ax=axes[0, 1], color='darkorange')
axes[0, 1].set_title('Test ROC-AUC', fontsize=14, fontweight='bold')
axes[0, 1].set_ylabel('ROC-AUC', fontsize=12)
axes[0, 1].tick_params(axis='x', rotation=45)
axes[0, 1].axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Random')
axes[0, 1].legend()

# Test Precision
results_df['test_precision'].plot(kind='bar', ax=axes[0, 2], color='forestgreen')
axes[0, 2].set_title('Test Precision', fontsize=14, fontweight='bold')
axes[0, 2].set_ylabel('Precision', fontsize=12)
axes[0, 2].tick_params(axis='x', rotation=45)

# Test Recall
results_df['test_recall'].plot(kind='bar', ax=axes[1, 0], color='crimson')
axes[1, 0].set_title('Test Recall', fontsize=14, fontweight='bold')
axes[1, 0].set_ylabel('Recall', fontsize=12)
axes[1, 0].tick_params(axis='x', rotation=45)

# Test F1 Score
results_df['test_f1'].plot(kind='bar', ax=axes[1, 1], color='purple')
axes[1, 1].set_title('Test F1 Score', fontsize=14, fontweight='bold')
axes[1, 1].set_ylabel('F1 Score', fontsize=12)
axes[1, 1].tick_params(axis='x', rotation=45)

# Overfitting Analysis (Train vs Test Accuracy)
overfitting = (results_df['train_accuracy'] - results_df['test_accuracy']) * 100
overfitting.plot(kind='bar', ax=axes[1, 2], color='coral')
axes[1, 2].set_title('Overfitting (Train-Test Accuracy %)', fontsize=14, fontweight='bold')
axes[1, 2].set_ylabel('Difference (%)', fontsize=12)
axes[1, 2].tick_params(axis='x', rotation=45)
axes[1, 2].axhline(y=0, color='black', linestyle='-', alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'model_performance_dashboard.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Dashboard saved to: {RESULTS_DIR / 'model_performance_dashboard.png'}")

In [ ]:
# Radar chart for model comparison
print("=== Radar Chart Comparison ===")

from math import pi

# Select metrics for radar chart
metrics = ['test_accuracy', 'test_precision', 'test_recall', 'test_f1', 'test_roc_auc']
model_names = results_df.index.tolist()

# Number of variables
N = len(metrics)

# Compute angle for each axis
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]  # Complete the circle

# Create radar chart
fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection='polar'))

colors = plt.cm.Set3(np.linspace(0, 1, len(model_names)))

for idx, model in enumerate(model_names):
    values = results_df.loc[model, metrics].values.flatten().tolist()
    values += values[:1]  # Complete the circle
    ax.plot(angles, values, 'o-', linewidth=2, label=model, color=colors[idx])
    ax.fill(angles, values, alpha=0.15, color=colors[idx])

# Add labels
ax.set_xticks(angles[:-1])
ax.set_xticklabels([m.replace('test_', '').title() for m in metrics], fontsize=11)
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=9)
ax.set_title('Model Performance Radar Chart', fontsize=16, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=10)
ax.grid(True)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'model_radar_chart.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Radar chart saved to: {RESULTS_DIR / 'model_radar_chart.png'}")

In [ ]:
# Load best model and create detailed analysis
print("=== Loading Best Model ===")

best_model_name = results_df['test_roc_auc'].idxmax()
print(f"Best model: {best_model_name}")

best_model_path = MODELS_DIR / f"{best_model_name.lower().replace(' ', '_')}.pkl"
with open(best_model_path, 'rb') as f:
    best_model = pickle.load(f)

print(f"Loaded model from: {best_model_path}")

In [ ]:
# Load test data for detailed analysis
print("=== Loading Test Data ===")

X_test = pd.read_parquet(PROCESSED_DIR / "test.parquet")
y_test = pd.read_parquet(PROCESSED_DIR / "test_labels.parquet").squeeze()

customer_id_test = X_test['CustomerID']
X_test_features = X_test.drop(columns=['CustomerID'])

print(f"Test features: {X_test_features.shape}")
print(f"Test labels: {y_test.shape}")

In [ ]:
# Generate predictions and create analysis dataframe
print("=== Generating Predictions ===")

from sklearn.metrics import roc_auc_score, roc_curve

y_test_pred = best_model.predict(X_test_features)
y_test_proba = best_model.predict_proba(X_test_features)[:, 1]

# Create analysis dataframe
analysis_df = pd.DataFrame({
    'CustomerID': customer_id_test,
    'Actual_Churn': y_test,
    'Predicted_Churn': y_test_pred,
    'Churn_Probability': y_test_proba
})

print(f"Analysis dataframe shape: {analysis_df.shape}")
print(analysis_df.head())

In [ ]:
# Prediction probability distribution
print("=== Prediction Probability Distribution ===")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Overall probability distribution
sns.histplot(data=analysis_df, x='Churn_Probability', bins=50, ax=axes[0], kde=True)
axes[0].set_title('Distribution of Churn Probabilities', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Churn Probability', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].axvline(x=0.5, color='red', linestyle='--', alpha=0.7, label='Threshold (0.5)')
axes[0].legend()

# Probability distribution by actual churn
sns.boxplot(data=analysis_df, x='Actual_Churn', y='Churn_Probability', ax=axes[1])
axes[1].set_title('Churn Probability by Actual Churn Status', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Actual Churn (0=No, 1=Yes)', fontsize=12)
axes[1].set_ylabel('Churn Probability', fontsize=12)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'probability_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Probability distribution saved to: {RESULTS_DIR / 'probability_distribution.png'}")

In [ ]:
# Confusion Matrix with percentages
print("=== Confusion Matrix Analysis ===")

from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_test_pred)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['No Churn', 'Churn'],
            yticklabels=['No Churn', 'Churn'])
axes[0].set_title('Confusion Matrix (Counts)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('True Label', fontsize=12)
axes[0].set_xlabel('Predicted Label', fontsize=12)

# Percentages
sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='Blues', ax=axes[1],
            xticklabels=['No Churn', 'Churn'],
            yticklabels=['No Churn', 'Churn'])
axes[1].set_title('Confusion Matrix (Percentages)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('True Label', fontsize=12)
axes[1].set_xlabel('Predicted Label', fontsize=12)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Confusion matrix saved to: {RESULTS_DIR / 'confusion_matrix.png'}")
print(f"\nConfusion Matrix Summary:")
print(f"True Negatives: {cm[0, 0]} ({cm_normalized[0, 0]:.2%})")
print(f"False Positives: {cm[0, 1]} ({cm_normalized[0, 1]:.2%})")
print(f"False Negatives: {cm[1, 0]} ({cm_normalized[1, 0]:.2%})")
print(f"True Positives: {cm[1, 1]} ({cm_normalized[1, 1]:.2%})")

In [ ]:
# ROC Curve with optimal threshold
print("=== ROC Curve Analysis ===")

fpr, tpr, thresholds = roc_curve(y_test, y_test_proba)
roc_auc = roc_auc_score(y_test, y_test_proba)

# Find optimal threshold (Youden's J statistic)
j_scores = tpr - fpr
optimal_idx = np.argmax(j_scores)
optimal_threshold = thresholds[optimal_idx]

plt.figure(figsize=(10, 8))
plt.plot(fpr, tpr, linewidth=2, label=f'{best_model_name} (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Random Classifier')
plt.scatter(fpr[optimal_idx], tpr[optimal_idx], marker='o', color='red', s=100, 
            label=f'Optimal Threshold = {optimal_threshold:.3f}')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve with Optimal Threshold', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, alpha=0.3)
plt.xlim([0, 1])
plt.ylim([0, 1.05])

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'roc_curve.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"ROC curve saved to: {RESULTS_DIR / 'roc_curve.png'}")
print(f"Optimal threshold: {optimal_threshold:.4f}")
print(f"ROC-AUC: {roc_auc:.4f}")

In [ ]:
# Feature importance visualization
print("=== Feature Importance Analysis ===")

if hasattr(best_model, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'feature': X_test_features.columns,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    
    # Top 15 features
    top_features = feature_importance.head(15)
    sns.barplot(x='importance', y='feature', data=top_features, ax=axes[0], palette='viridis')
    axes[0].set_title(f'Top 15 Feature Importances - {best_model_name}', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Importance', fontsize=12)
    axes[0].set_ylabel('Feature', fontsize=12)
    
    # Cumulative importance
    cumulative_importance = np.cumsum(feature_importance['importance'])
    n_features_90 = np.argmax(cumulative_importance >= 0.9) + 1
    
    axes[1].plot(range(1, len(cumulative_importance) + 1), cumulative_importance, linewidth=2)
    axes[1].axhline(y=0.9, color='red', linestyle='--', alpha=0.7, label='90% threshold')
    axes[1].axvline(x=n_features_90, color='green', linestyle='--', alpha=0.7, 
                label=f'{n_features_90} features')
    axes[1].set_title('Cumulative Feature Importance', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Number of Features', fontsize=12)
    axes[1].set_ylabel('Cumulative Importance', fontsize=12)
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'feature_importance.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"Feature importance saved to: {RESULTS_DIR / 'feature_importance.png'}")
    print(f"\nTop 10 features:")
    print(feature_importance.head(10).to_string(index=False))
    print(f"\n{n_features_90} features explain 90% of the importance")
    
elif hasattr(best_model, 'coef_'):
    # For logistic regression
    feature_importance = pd.DataFrame({
        'feature': X_test_features.columns,
        'coefficient': best_model.coef_[0]
    }).sort_values('coefficient', key=abs, ascending=False)
    
    plt.figure(figsize=(12, 8))
    top_features = feature_importance.head(15)
    colors = ['red' if x < 0 else 'green' for x in top_features['coefficient']]
    top_features['coefficient'].plot(kind='barh', color=colors)
    plt.title(f'Top 15 Feature Coefficients - {best_model_name}', fontsize=14, fontweight='bold')
    plt.xlabel('Coefficient', fontsize=12)
    plt.ylabel('Feature', fontsize=12)
    plt.axvline(x=0, color='black', linestyle='-', alpha=0.3)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'feature_importance.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"Feature importance saved to: {RESULTS_DIR / 'feature_importance.png'}")
else:
    print("Feature importance not available for this model")

In [ ]:
# Save analysis results
print("=== Saving Analysis Results ===")

# Save analysis dataframe
analysis_df.to_csv(RESULTS_DIR / 'prediction_analysis.csv', index=False)
print(f"Prediction analysis saved to: {RESULTS_DIR / 'prediction_analysis.csv'}")

# Create summary report
summary_report = f"""
# Telco Churn Prediction - Model Results Summary

## Best Model
- Model: {best_model_name}
- Test ROC-AUC: {results_df.loc[best_model_name, 'test_roc_auc']:.4f}
- Test Accuracy: {results_df.loc[best_model_name, 'test_accuracy']:.4f}
- Test Precision: {results_df.loc[best_model_name, 'test_precision']:.4f}
- Test Recall: {results_df.loc[best_model_name, 'test_recall']:.4f}
- Test F1 Score: {results_df.loc[best_model_name, 'test_f1']:.4f}

## Model Comparison
{results_df.round(4).to_string()}

## Key Findings
- Best performing model: {best_model_name}
- All models achieved ROC-AUC > 0.5
- Optimal classification threshold: {optimal_threshold:.4f}
- Consider class imbalance handling for further improvement

## Generated Visualizations
- model_performance_dashboard.png
- model_radar_chart.png
- probability_distribution.png
- confusion_matrix.png
- roc_curve.png
- feature_importance.png
"""

with open(RESULTS_DIR / 'summary_report.md', 'w') as f:
    f.write(summary_report)

print(f"Summary report saved to: {RESULTS_DIR / 'summary_report.md'}")

In [ ]:
# Final summary
print("=== Results Generation Complete ===")
print(f"\nAll results saved to: {RESULTS_DIR}")
print(f"\nGenerated files:")
for file in RESULTS_DIR.glob('*'):
    print(f"  - {file.name}")

print(f"\nNext steps:")
print(f"1. Review visualizations in {RESULTS_DIR}")
print(f"2. Analyze prediction_analysis.csv for detailed predictions")
print(f"3. Read summary_report.md for comprehensive summary")
print(f"4. Consider hyperparameter tuning for improved performance")